# Solve a heat equation with Skeleton-KAN

Solve $u_t-0.1u_{xx}=0$ on $[0,1]^2$, with $u(x,0)=\sin(\pi x)$ and $u(0,t)=u(1,t)=0$. The reference is $u(x,t)=e^{-0.1\pi^2t}\sin(\pi x)$.

This notebook uses the **project standalone L-BFGS**, not `torch.optim.LBFGS` or seed batching. It instantiates one pre-existing **KS-IES** skeleton (`IAPB13`) from all 300 source equations. The frozen bank has no access to this target's observations during construction. The notebook does not perform the paper's 18-candidate, ten-seed benchmark; it illustrates the single-model API. Its data are independently sampled with seed 421. The default is 50 outer calls, at most 20 inner iterations each, in float64. Change the settings in the next cell for a shorter launch check.

The supplied source bank's symbolic-regression domain is **VSR-DPG modified Livermore2**. Frozen filenames retain `v2` only as a compatibility key for KS-IES.

In [ ]:
from pathlib import Path
from types import SimpleNamespace
from datetime import datetime, timezone
import copy
import json
import math
import sys
import torch

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "structured_kan").is_dir())
sys.path.insert(0, str(ROOT))
from examples.common import build_model, metrics, save, standardization
from structured_kan.model.StructuredKANBuilder import StructuredKANBuilder
from structured_kan.optimizer.lbfgs import LBFGS

args = SimpleNamespace(
    device="cuda" if torch.cuda.is_available() else "cpu",
    seed=421, builder="IAPB13", points=1024, outer_steps=50,
    output=ROOT / "structured_kan/data/results" /
        ("notebook_solve_pde_" + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")),
)
print({"device": args.device, "torch": torch.__version__,
       "optimizer": "project standalone LBFGS", "seed_batch": False})

## Sample data and standardize

Input statistics use collocation points; output statistics use prescribed initial/boundary values only. No interior reference-solution values enter the training loss.

In [ ]:
generator = torch.Generator(device='cpu').manual_seed(args.seed)
nu = 0.1

def sample(n):
    return torch.rand(n, 2, dtype=torch.float64, generator=generator).to(args.device)

def exact(xt):
    return torch.exp(-nu*math.pi**2*xt[:, 1:2])*torch.sin(math.pi*xt[:, :1])

interior = sample(args.points)
n_constraint = max(4, args.points//4)
initial, left, right = (sample(n_constraint) for _ in range(3))
initial[:, 1] = 0.
left[:, 0], right[:, 0] = 0., 1.
constraints = torch.cat([initial, left, right])
values = torch.cat([torch.sin(math.pi*initial[:, :1]),
                    left.new_zeros(n_constraint, 1), right.new_zeros(n_constraint, 1)])
validation, test = sample(args.points), sample(args.points)
x_mean, x_std = standardization(interior)
# Only prescribed IC/BC values determine output normalization.
y_mean, y_std = standardization(values)


## Materialize the frozen skeleton

`build_model` loads the frozen source-derived per-map budgets and creates private affine routes with source-role-shared unary maps. The training inputs calibrate the fixed grids. The skeleton topology is not fitted to this equation.

In [ ]:
model, spec, metadata = build_model(args, (interior-x_mean)/x_std)

print({k: metadata[k] for k in ["constructor", "builder", "parameters", "bank_sha256"]})

## Standalone optimizer and closure

Each closure recomputes the complete loss and calls `backward()`. There is one model and one L-BFGS history. Strong-Wolfe trials use this same closure.

In [ ]:
optimizer = LBFGS(
    model.parameters(), lr=1., max_iter=20, history_size=100,
    tolerance_grad=1e-32, tolerance_change=1e-32, tolerance_ys=1e-32,
    line_search_fn="strong_wolfe", two_loop_mode="nosync_fma",
    scalar_mode="coalesced_host",
)

def predict(xt):
    return model((xt-x_mean)/x_std)*y_std + y_mean

def residual(xt):
    # Differentiation includes normalization, so derivatives are physical.
    xt = xt.detach().requires_grad_(True)
    u = predict(xt)
    grad = torch.autograd.grad(u.sum(), xt, create_graph=True)[0]
    u_t, u_x = grad[:, 1:2], grad[:, :1]
    u_xx = torch.autograd.grad(u_x.sum(), xt, create_graph=True)[0][:, :1]
    return u_t - nu*u_xx

def closure():
    optimizer.zero_grad(set_to_none=True)
    loss = (0.01*residual(interior).square().mean()
            + (predict(constraints)-values).square().mean()) / y_std.square().squeeze()
    loss.backward()
    return loss

@torch.no_grad()
def validation_loss():
    return metrics(predict(validation), exact(validation))['nmse']

## Train and retain the best validation checkpoint

For this analytic example, exact interior values are used only for checkpoint selection and final reporting. This is not a reference-free PDE-selection protocol.

In [ ]:
best_loss = float(validation_loss())
best_state = copy.deepcopy(model.state_dict())
best_step = 0
for step in range(1, args.outer_steps + 1):
    optimizer.step(closure)
    loss = float(validation_loss())
    if not math.isfinite(loss):
        raise FloatingPointError("non-finite validation error")
    if loss < best_loss:
        best_loss, best_state, best_step = loss, copy.deepcopy(model.state_dict()), step
    if step == 1 or step % 10 == 0 or step == args.outer_steps:
        print(f"outer={step:2d}, validation NMSE={loss:.6e}")
model.load_state_dict(best_state)
print({"selected_outer_step": best_step})

## Report original-scale MSE and normalized MSE; save the model

In [ ]:
residual_mse = float(residual(test).square().mean().detach())
with torch.no_grad():
    result = dict(task='heat_pde', pde='u_t - 0.1*u_xx = 0',
        exact_solution='exp(-0.1*pi**2*t)*sin(pi*x)',
        support=[[0., 1.], [0., 1.]], interior_points=args.points,
        points_per_constraint=n_constraint, selected_outer_step=best_step,
        residual_weight=0.01, interior_training_labels=False,
        test_residual_mse=residual_mse,
        validation=metrics(predict(validation), exact(validation)),
        test=metrics(predict(test), exact(test)))
save(args, model, spec, metadata,
     dict(x_mean=x_mean, x_std=x_std, y_mean=y_mean, y_std=y_std), result)


## Verify the saved checkpoint

In [ ]:
checkpoint = torch.load(args.output / "model.pt", map_location=args.device, weights_only=True)
restored = StructuredKANBuilder(2, dtype=torch.float64, device=args.device).build(checkpoint["spec"])
restored.load_state_dict(checkpoint["state_dict"])
stats = checkpoint["normalizers"]
with torch.no_grad():
    check_x = test
    restored_prediction = restored((check_x-stats["x_mean"])/stats["x_std"])*stats["y_std"] + stats["y_mean"]
    torch.testing.assert_close(restored_prediction, predict(check_x), rtol=0., atol=0.)
print("Saved checkpoint reproduces predictions exactly.")

## Visualize held-out predictions

In [ ]:
import matplotlib.pyplot as plt
axis = torch.linspace(0, 1, 81, device=args.device, dtype=torch.float64)
xx, tt = torch.meshgrid(axis, axis, indexing="ij")
grid = torch.stack([xx.flatten(), tt.flatten()], dim=1)
with torch.no_grad():
    truth = exact(grid).reshape(81,81).cpu().numpy()
    learned = predict(grid).reshape(81,81).cpu().numpy()
fig, axes = plt.subplots(1, 3, figsize=(11, 3.2))
for ax, values, title in zip(axes, [truth, learned, abs(learned-truth)],
                            ["Exact solution", "Learned solution", "Absolute error"]):
    im = ax.imshow(values, origin="lower", extent=[0,1,0,1], aspect="auto")
    ax.set(xlabel="t", ylabel="x", title=title)
    fig.colorbar(im, ax=ax)
fig.tight_layout()
plt.show()